# Week 7：Gradient Boosting 基础

目标：用分类任务观察 Boosting 中 `learning_rate` 与 `n_estimators` 的配合。要求理解“后一棵树修正当前误差”的机制；不要求手推完整梯度提升或 XGBoost 目标函数。

In [1]:
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split

In [2]:
dataset = load_breast_cancer(as_frame=True)
X = dataset.data
y = dataset.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## 1. 学习率与树数量的配合

每一组都使用浅树 `max_depth=2`。实验只改变 `learning_rate` 和 `n_estimators`：学习率变小时，通常需要更多树才能完成足够的修正。此阶段只使用训练集进行交叉验证，`X_test` 不参与实验。

In [3]:
settings = [
    {'learning_rate': 0.30, 'n_estimators': 30},
    {'learning_rate': 0.10, 'n_estimators': 100},
    {'learning_rate': 0.03, 'n_estimators': 300},
]

rows = []
for setting in settings:
    model = GradientBoostingClassifier(
        learning_rate=setting['learning_rate'],
        n_estimators=setting['n_estimators'],
        max_depth=2,
        random_state=42,
    )

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring='accuracy',
        return_train_score=True,
    )

    rows.append({
        **setting,
        'train_mean': scores['train_score'].mean(),
        'validation_mean': scores['test_score'].mean(),
        'validation_std': scores['test_score'].std(),
    })

boosting_results = pd.DataFrame(rows)
display(boosting_results.round(3))

,learning_rate,n_estimators,train_mean,validation_mean,validation_std
0,0.30,30,1.0,0.956,0.018
1,0.10,100,1.0,0.963,0.013
2,0.03,300,1.0,0.960,0.015


## 2. 如何读结果

优先比较 `validation_mean`；当它们接近时，再比较 `validation_std`、训练与验证的差距、训练成本和模型复杂度。不要只根据训练准确率选择参数。

## 检查点

1. 为什么学习率从 0.30 降到 0.03 时，通常要增加树数量？
2. 如果树数量很多、学习率也很大，可能有什么风险？
3. 这三组参数中，应依据哪些列做选择？